In [1]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
import numpy as np
import torchvision
from tqdm import tqdm

/home/shkaf2m/Desktop/ml-isp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
def custom_collate_function(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]
  labels = torch.tensor(labels, dtype=torch.long)
  return images, labels

val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
])
val_dataset = Imagenette(root = './data', split = 'val', download = True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False, num_workers = 7, collate_fn = custom_collate_function)

In [10]:
def ValidateModel(model, device, data_loader, text_features):
  true_labels = torch.tensor([]).to(device)
  predicted_labels = torch.tensor([]).to(device)

  text_features = text_features / text_features.norm(dim=-1, keepdim=True)
  logit_scale = model.logit_scale.exp()

  with torch.no_grad():
    for images, labels in tqdm(data_loader):
      labels = labels.to(device)
      image_inputs = processor(images = images, return_tensors = "pt", padding = True).to(device)
      image_features = model.get_image_features(**image_inputs)
       # Косинусное сходство между image features и text features + масштабирование
      similarity = (image_features @ text_features.T) * logit_scale

      predicted = similarity.argmax(dim=1)
      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)

  return true_labels, predicted_labels

In [11]:
import os
from sklearn.metrics import f1_score
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model = model.to(DEVICE)
model.eval()

all_classes = [label[0] for label in val_dataset.classes]
text_inputs = processor(text = all_classes, return_tensors = "pt", padding = True).to(DEVICE)
text_features = model.get_text_features(**text_inputs)
true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)

f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')
print("F1 Score: ", f1_res)


100%|██████████| 123/123 [00:35<00:00,  3.44it/s]

F1 Score:  0.9876700606902598


Попробуем несколько подходов:
1) Используем названия классов
2) Используем "a photo of {название класса}"
3) Для каждого класса подберем подходящие слова

In [12]:
prompts = [label[0] for label in val_dataset.classes]
print("Promts: ", prompts)

text_inputs = processor(text = prompts, return_tensors = "pt", padding = True).to(DEVICE)
text_features = model.get_text_features(**text_inputs)
true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)

f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')
print("F1 Score: ", f1_res)

Promts:  ['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']


100%|██████████| 123/123 [00:35<00:00,  3.44it/s]

F1 Score:  0.9876700606902598


In [13]:
prompts = [f"a photo of {label[0]}" for label in val_dataset.classes]
print("Promts: ", prompts)

text_inputs = processor(text = prompts, return_tensors = "pt", padding = True).to(DEVICE)
text_features = model.get_text_features(**text_inputs)
true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)

f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')
print("F1 Score: ", f1_res)

Promts:  ['a photo of tench', 'a photo of English springer', 'a photo of cassette player', 'a photo of chain saw', 'a photo of church', 'a photo of French horn', 'a photo of garbage truck', 'a photo of gas pump', 'a photo of golf ball', 'a photo of parachute']


100%|██████████| 123/123 [00:36<00:00,  3.37it/s]

F1 Score:  0.9835235748426783


In [15]:
prompts = ["slimy tench", "fast English springer", "good cassette player", "electric chain saw", "religious church", "French horn", "garbage truck", "gas pump", "white golf ball", "chute parachute"]
print("Promts: ", prompts)

text_inputs = processor(text = prompts, return_tensors = "pt", padding = True).to(DEVICE)
text_features = model.get_text_features(**text_inputs)
true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)

f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')
print("F1 Score: ", f1_res)

Promts:  ['slimy tench', 'fast English springer', 'good cassette player', 'electric chain saw', 'religious church', 'French horn', 'garbage truck', 'gas pump', 'white golf ball', 'chute parachute']


  0%|          | 0/123 [00:00<?, ?it/s]

100%|██████████| 123/123 [00:35<00:00,  3.43it/s]

F1 Score:  0.9817688668017928


Теперь, попробуем усреднить эмбендинги текстовых промтов

In [16]:
prompts_1 = [label[0] for label in val_dataset.classes]
prompts_2 = [f"a photo of {label[0]}" for label in val_dataset.classes]
prompts_3 = ["slimy tench", "fast English springer", "good cassette player", "electric chain saw", "religious church", "French horn", "garbage truck", "gas pump", "white golf ball", "chute parachute"]

text_inputs_1 = processor(text=prompts_1, return_tensors="pt", padding=True).to(DEVICE)
text_inputs_2 = processor(text=prompts_2, return_tensors="pt", padding=True).to(DEVICE)
text_inputs_3 = processor(text=prompts_3, return_tensors="pt", padding=True).to(DEVICE)

text_features_1 = model.get_text_features(**text_inputs_1)
text_features_2 = model.get_text_features(**text_inputs_2)
text_features_3 = model.get_text_features(**text_inputs_3)

text_features_avg = (text_features_1 + text_features_2 + text_features_3) / 3

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features_avg)

f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')
print("F1 Score: ", f1_res)

100%|██████████| 123/123 [00:35<00:00,  3.49it/s]

F1 Score:  0.9863541483631367
